# Cloud Provider Analytics — MVP (Segundo Parcial)
**Big Data 72.80 · ITBA · Lambda Architecture · PySpark + Structured Streaming + Cassandra/AstraDB**

Camila Lee (63382) · Lucas Perri (62746)

End-to-end minimum: `Landing → Bronze → Silver → Gold → Cassandra`. Each section names the **CLAUDE.md §** it implements and a one-line *why* for the video.

### MVP checklist
- [ ] Batch → Bronze: 3 masters, partitioned Parquet, `ingest_ts`+`source_file`, dedupe
- [ ] Streaming → Bronze: explicit schema, `withWatermark`, dedupe by `event_id`, late data, checkpoint
- [ ] Silver: clean + enrich + ≥3 features + 3 quality rules + quarantine + anomaly flag
- [ ] Gold: `org_daily_usage_by_service` (daily FinOps mart)
- [ ] Serving: query-first **CQL table** (real Cassandra), loaded from Spark
- [ ] Q1 + Q2 with results (CQL shown next to each)
- [ ] Idempotency: re-run without duplicating (counts before/after)

### Design choices that differentiate this (say these in the video)
- **Real CQL tables** with `PRIMARY KEY ((org_id, service), usage_date)` — query-first modeling with explicit partition/clustering keys, **not** schemaless Document-API collections.
- **Metric-based features:** `requests` = `value` where `metric='requests'` (a `count(*)` of rows would be wrong).
- **`value` read as string then cast** — it arrives as number, `"100.0"`, or null.
- **Per-service anomalies, flagged only when ≥2 of 3 methods agree** — avoids the 'half the rows are anomalies' noise of a global OR.
- **Real costs preserved** (we flag, we don't clamp).

## 0. Setup — AstraDB (once, before running)
*Why:* Cassandra is the serving layer; AstraDB is hosted Cassandra. Connection = **Secure Connect Bundle (SCB)** + **application token**.
1. astra.datastax.com → create a **Serverless (Non-Vector)** DB; create keyspace **`cloud_analytics`** in the UI (can't `CREATE KEYSPACE` from the driver on serverless; `CREATE TABLE` works).
2. Generate an **application token** (role *Database Administrator*) → copy the `AstraCS:...` string.
3. **Download the Secure Connect Bundle** zip; in Colab open **Files → Upload** it to `/content/`.
4. Upload the dataset so `/content/datalake/landing/` holds the 7 CSVs and `usage_events_stream/*.jsonl`.

`/content` is wiped between sessions (fine for one demo run). Mount Drive to persist.

In [ ]:
# --- 1. Installs ---
!pip -q install pyspark==3.5.1 cassandra-driver
print("installed")

In [ ]:
# --- 2. Spark session | §4.1, §4.3 ---
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType, DoubleType,
                               LongType, IntegerType, BooleanType, DateType)
spark = (SparkSession.builder.appName("cloud_provider_analytics_mvp")
         .master("local[*]").config("spark.sql.shuffle.partitions","8").getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark", spark.version)

In [ ]:
# --- 3. Paths / zones | §3.3 ---
import os
BASE="/content/datalake"; LANDING=f"{BASE}/landing"
BRONZE=f"{BASE}/bronze"; SILVER=f"{BASE}/silver"; GOLD=f"{BASE}/gold"
QUARANTINE=f"{BASE}/quarantine"; CHK="/content/checkpoints"
for p in [BRONZE,SILVER,GOLD,QUARANTINE,CHK]: os.makedirs(p, exist_ok=True)
EVENTS_DIR=f"{LANDING}/usage_events_stream"
print("landing:", os.path.exists(LANDING), "| events:", os.path.exists(EVENTS_DIR))

## 1. Schemas | §2.2, §12.2
Explicit schemas (never `inferSchema`). Event schema is a **nullable union of v1+v2**; `value` is **string** (cast later with fallback); the timestamp field is literally `timestamp`.

In [ ]:
event_schema = StructType([
    StructField("event_id", StringType()), StructField("timestamp", StringType()),
    StructField("org_id", StringType()), StructField("resource_id", StringType()),
    StructField("service", StringType()), StructField("region", StringType()),
    StructField("metric", StringType()), StructField("value", StringType()),   # string -> cast
    StructField("unit", StringType()), StructField("cost_usd_increment", DoubleType()),
    StructField("schema_version", IntegerType()),
    StructField("carbon_kg", DoubleType()), StructField("genai_tokens", LongType()),  # v2 nullable
])
orgs_schema = StructType([
    StructField("org_id",StringType()),StructField("org_name",StringType()),
    StructField("industry",StringType()),StructField("hq_region",StringType()),
    StructField("plan_tier",StringType()),StructField("is_enterprise",BooleanType()),
    StructField("signup_date",DateType()),StructField("sales_rep",StringType()),
    StructField("lifecycle_stage",StringType()),StructField("marketing_source",StringType()),
    StructField("nps_score",DoubleType())])
users_schema = StructType([
    StructField("user_id",StringType()),StructField("org_id",StringType()),
    StructField("email",StringType()),StructField("role",StringType()),
    StructField("active",BooleanType()),StructField("created_at",DateType()),
    StructField("last_login",DateType())])
billing_schema = StructType([
    StructField("invoice_id",StringType()),StructField("org_id",StringType()),
    StructField("month",DateType()),StructField("subtotal",DoubleType()),
    StructField("credits",DoubleType()),StructField("taxes",DoubleType()),
    StructField("currency",StringType()),StructField("exchange_rate_to_usd",DoubleType())])
print("schemas ready")

## 2. Batch → Bronze | §7.1, §12.3
Typed, deduped, provenance columns, partitioned Parquet.

In [ ]:
def csv_to_bronze(path, schema, key, out):
    df = (spark.read.option("header",True).schema(schema).csv(path)
          .withColumn("ingest_ts",F.current_timestamp())
          .withColumn("source_file",F.input_file_name())
          .dropDuplicates([key]))
    df.write.mode("overwrite").parquet(out); print(f"{out}: {df.count()} rows"); return df

orgs    = csv_to_bronze(f"{LANDING}/customers_orgs.csv", orgs_schema,   "org_id",    f"{BRONZE}/orgs")
users   = csv_to_bronze(f"{LANDING}/users.csv",          users_schema,  "user_id",   f"{BRONZE}/users")
billing = csv_to_bronze(f"{LANDING}/billing_monthly.csv",billing_schema,"invoice_id",f"{BRONZE}/billing")

## 3. Streaming → Bronze (Speed-Layer ingestion) | §5, §12.2
Unbounded table → cast `timestamp`/`value` → watermark + dedupe by `event_id` → Parquet + checkpoint, `availableNow` (process all files then stop).
> `readStream` builds nothing; **`writeStream.start()`** runs.

In [ ]:
raw = spark.readStream.schema(event_schema).json(EVENTS_DIR)
bronze_stream = (raw
    .withColumn("event_ts",  F.to_timestamp("timestamp"))
    .withColumn("value_num", F.col("value").cast("double"))   # "100.0"/99.0 -> double; junk -> null
    .withColumn("usage_date",F.to_date("event_ts"))
    .withColumn("ingest_ts", F.current_timestamp())
    .withColumn("source_file", F.input_file_name())
    .withWatermark("event_ts","10 minutes")
    .dropDuplicates(["event_id"]))
q = (bronze_stream.writeStream.format("parquet")
     .option("path", f"{BRONZE}/events")
     .option("checkpointLocation", f"{CHK}/bronze_events")
     .outputMode("append").trigger(availableNow=True).start())
q.awaitTermination()
print("streaming->bronze done")

In [ ]:
bronze_events = spark.read.parquet(f"{BRONZE}/events")
print("bronze events:", bronze_events.count())
bronze_events.select("event_id","event_ts","org_id","service","metric","value","value_num",
                     "cost_usd_increment","schema_version","genai_tokens").show(5, truncate=False)

## 4. Silver — quality + quarantine + enrichment | §8.1, §12.4
3 rules → failures to **quarantine**; a final batch `dropDuplicates` (global net beyond the watermark window); broadcast-join the org dimension.

In [ ]:
ev = spark.read.parquet(f"{BRONZE}/events").dropDuplicates(["event_id"])
valid = (F.col("event_id").isNotNull()
         & (F.col("cost_usd_increment") >= -0.01)
         & ~(F.col("value").isNotNull() & F.col("unit").isNull()))   # rule 3
silver_ok  = ev.filter(valid)
quarantine = ev.filter(~valid)
quarantine.write.mode("overwrite").parquet(f"{QUARANTINE}/events")
print("valid:", silver_ok.count(), "| quarantined:", quarantine.count())
quarantine.select("event_id","value","unit","cost_usd_increment").show(5, truncate=False)

In [ ]:
orgs_dim = spark.read.parquet(f"{BRONZE}/orgs").select(
    "org_id","industry","plan_tier","hq_region","lifecycle_stage")
silver = silver_ok.join(F.broadcast(orgs_dim), on="org_id", how="left")
print("silver enriched:", silver.count())

### 4b. Anomaly detection — z-score / MAD / p99, flag on ≥2 | §8.2
Per-service stats (costs differ by service). Flag only when **≥2 of 3** agree → low false positives. We **flag, never clamp** — real costs stay intact.

In [ ]:
stats = silver.groupBy("service").agg(
    F.mean("cost_usd_increment").alias("mu"), F.stddev("cost_usd_increment").alias("sd"),
    F.expr("percentile_approx(cost_usd_increment,0.5)").alias("med"),
    F.expr("percentile_approx(cost_usd_increment,0.99)").alias("p99"))
s1 = silver.join(F.broadcast(stats), on="service", how="left") \
           .withColumn("abs_dev", F.abs(F.col("cost_usd_increment")-F.col("med")))
mad = s1.groupBy("service").agg(F.expr("percentile_approx(abs_dev,0.5)").alias("mad"))
s2  = s1.join(F.broadcast(mad), on="service", how="left")

K = 1.5  # p99 factor (threshold decision -> decision_log)
z_flag   = (F.abs(F.col("cost_usd_increment")-F.col("mu"))/F.col("sd")) > 3
mad_flag = (F.col("mad")>0) & ((F.col("abs_dev")/(1.4826*F.col("mad"))) > 3)
p99_flag = F.col("cost_usd_increment") > (F.col("p99")*K)
silver_flagged = s2.withColumn("anomaly",
    (z_flag.cast("int")+mad_flag.cast("int")+p99_flag.cast("int")) >= 2)

silver_flagged.write.mode("overwrite").partitionBy("usage_date","service").parquet(f"{SILVER}/events")
print("anomalies flagged:", silver_flagged.filter("anomaly").count())
silver_flagged.filter("anomaly").select(
    "event_id","org_id","service","cost_usd_increment","mu","p99").orderBy(
    F.desc("cost_usd_increment")).show(10, truncate=False)
silver_flagged.filter("anomaly").groupBy("service").count().show()

## 5. Gold — daily FinOps mart | §7.2, §12.6
Grain org×service×day. **Features from `metric`**; money **rounded** at the serving boundary.

In [ ]:
se = spark.read.parquet(f"{SILVER}/events")
gold_finops = (se.groupBy("org_id","service","usage_date").agg(
        F.round(F.sum("cost_usd_increment"),2).alias("daily_cost_usd"),
        F.round(F.sum(F.when(F.col("metric")=="requests", F.col("value_num"))),2).alias("requests"),
        F.round(F.sum(F.when(F.col("metric")=="cpu_hours", F.col("value_num"))),4).alias("cpu_hours"),
        F.round(F.sum(F.when(F.col("metric")=="storage_gb_hours", F.col("value_num"))),4).alias("storage_gb_hours"),
        F.sum("genai_tokens").alias("genai_tokens"),
        F.round(F.sum("carbon_kg"),6).alias("carbon_kg"))
    .fillna(0, subset=["requests","cpu_hours","storage_gb_hours","genai_tokens","carbon_kg"]))
gold_finops.write.mode("overwrite").partitionBy("usage_date").parquet(f"{GOLD}/org_daily_usage_by_service")
print("gold rows:", gold_finops.count())
gold_finops.orderBy(F.desc("daily_cost_usd")).show(8, truncate=False)

## 6. Serving — Cassandra/AstraDB (CQL table) | §6, §12.8
Query-first: `PRIMARY KEY ((org_id, service), usage_date)`. UPSERT = idempotent. **`dict_factory`** so rows support `r["col"]`.

In [ ]:
from cassandra.cluster import Cluster
from cassandra.auth import PlainTextAuthProvider
from cassandra.query import dict_factory

SCB_PATH    = "/content/secure-connect-YOURDB.zip"   # <-- edit
ASTRA_TOKEN = "AstraCS:xxxxxxxx"                      # <-- edit
KEYSPACE    = "cloud_analytics"                       # <-- keyspace created in UI

cluster = Cluster(cloud={"secure_connect_bundle": SCB_PATH},
                  auth_provider=PlainTextAuthProvider("token", ASTRA_TOKEN))
session = cluster.connect(KEYSPACE)
session.row_factory = dict_factory                   # rows behave like dicts
print("connected:", session.execute("select release_version from system.local").one())

In [ ]:
session.execute(f'''
CREATE TABLE IF NOT EXISTS {KEYSPACE}.org_daily_usage_by_service (
  org_id text, service text, usage_date date,
  daily_cost_usd double, requests double, cpu_hours double, storage_gb_hours double,
  genai_tokens bigint, carbon_kg double,
  PRIMARY KEY ((org_id, service), usage_date)
) WITH CLUSTERING ORDER BY (usage_date DESC)''')
print("table ready")

In [ ]:
import datetime
upsert = session.prepare(f'''
INSERT INTO {KEYSPACE}.org_daily_usage_by_service
(org_id, service, usage_date, daily_cost_usd, requests, cpu_hours, storage_gb_hours, genai_tokens, carbon_kg)
VALUES (?,?,?,?,?,?,?,?,?)''')

def load_finops():
    n=0
    for r in gold_finops.toLocalIterator():
        ud=r["usage_date"]
        if isinstance(ud, datetime.datetime): ud=ud.date()
        session.execute(upsert,(r["org_id"], r["service"], ud,
            float(r["daily_cost_usd"] or 0), float(r["requests"] or 0),
            float(r["cpu_hours"] or 0), float(r["storage_gb_hours"] or 0),
            int(r["genai_tokens"] or 0), float(r["carbon_kg"] or 0)))
        n+=1
    return n
print("upserted rows:", load_finops())

## 7. Queries Q1 & Q2 | §9
Date ranges are derived **from the data itself** (no hard-coded year — avoids the 2025-vs-2026 trap). Each query prints its CQL next to the result.

In [ ]:
import pandas as pd, datetime
sample = gold_finops.orderBy(F.desc("daily_cost_usd")).first()
ORG, SVC = sample["org_id"], sample["service"]
bounds = gold_finops.agg(F.min("usage_date").alias("lo"), F.max("usage_date").alias("hi")).first()
LO, HI = bounds["lo"], bounds["hi"]
print(f"Q1 sample: org={ORG} service={SVC} range={LO}..{HI}")

q1_cql = f'''SELECT usage_date, daily_cost_usd, requests
FROM {KEYSPACE}.org_daily_usage_by_service
WHERE org_id='{ORG}' AND service='{SVC}'
  AND usage_date>='{LO}' AND usage_date<='{HI}';'''
print("-- Q1 (CQL)\n"+q1_cql)

rows = session.execute(f'''SELECT usage_date, daily_cost_usd, requests
  FROM {KEYSPACE}.org_daily_usage_by_service
  WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s''', (ORG,SVC,LO,HI))
pd.DataFrame(list(rows))

In [ ]:
cutoff = HI - datetime.timedelta(days=14)
print(f"-- Q2 (CQL, repeated per service) window {cutoff}..{HI}")
services = sorted({r["service"] for r in session.execute(
    f"SELECT org_id, service FROM {KEYSPACE}.org_daily_usage_by_service") if r["org_id"]==ORG})
totals=[]
for s in services:
    tot = sum((r["daily_cost_usd"] or 0) for r in session.execute(
        f'''SELECT daily_cost_usd FROM {KEYSPACE}.org_daily_usage_by_service
            WHERE org_id=%s AND service=%s AND usage_date>=%s AND usage_date<=%s''',
        (ORG, s, cutoff, HI)))
    totals.append((s, round(tot,4)))
pd.DataFrame(sorted(totals, key=lambda x:-x[1]), columns=["service","cost_14d"])

## 8. Idempotency proof | §8.4
UPSERT overwrites by key → re-running leaves the count unchanged.

In [ ]:
before = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
load_finops()
after  = session.execute(f"SELECT COUNT(*) AS c FROM {KEYSPACE}.org_daily_usage_by_service").one()["c"]
print(f"before={before} after={after} -> idempotent: {before==after}")

## 9. Observability + cleanup | §5.10
`lastProgress` from the ingestion query; stop streams when done.

In [ ]:
print("ingestion lastProgress:", q.lastProgress)
for s in spark.streams.active: s.stop()
print("active streams:", spark.streams.active)